# Ансамбли моделей машинного обучения. Часть 1.

In [1]:
import pandas as pd

### 1. Выберите набор данных (датасет) для решения задачи классификации или регресии.

Используем данные из Student Performance Prediction Dataset https://www.kaggle.com/datasets/shambhurajejagadale/student-performance-prediction-dataset/data

In [2]:
data = pd.read_csv('student_dataset_10000_rows.csv', sep=",")

In [3]:
# размер набора данных
data.shape

(10000, 8)

In [4]:
# типы колонок
data.dtypes

study_hours                int64
attendance                 int64
sleep_hours                int64
internet_usage             int64
assignments_completed      int64
previous_score             int64
exam_score               float64
placement_status             str
dtype: object

### 2. В случае необходимости проведите удаление или заполнение пропусков и кодирование категориальных признаков.

In [5]:
data.isnull().sum()

study_hours              0
attendance               0
sleep_hours              0
internet_usage           0
assignments_completed    0
previous_score           0
exam_score               0
placement_status         0
dtype: int64

In [6]:
data.head()

,study_hours,attendance,sleep_hours,internet_usage,assignments_completed,previous_score,exam_score,placement_status
0,7,56,8,7,10,62,100.00,Placed
1,4,69,5,3,8,56,100.00,Placed
2,11,60,7,6,10,45,100.00,Placed
3,8,99,9,8,4,55,90.17,Placed
4,5,52,8,6,8,40,78.82,Placed


Я выбрал регрессию по exam_score но у нас есть категориальный признак, закодирую ее с LabelEncoding

In [10]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
data['placement_status'] = le.fit_transform(data['placement_status'])

In [11]:
data.head()

,study_hours,attendance,sleep_hours,internet_usage,assignments_completed,previous_score,exam_score,placement_status
0,7,56,8,7,10,62,100.00,1
1,4,69,5,3,8,56,100.00,1
2,11,60,7,6,10,45,100.00,1
3,8,99,9,8,4,55,90.17,1
4,5,52,8,6,8,40,78.82,1


### 3. С использованием метода train_test_split разделите выборку на обучающую и тестовую.

In [15]:
from sklearn.model_selection import train_test_split

X = data.drop('exam_score', axis=1)
y = data['exam_score']

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("Размер обучающей выборки:", X_train.shape)
print("Размер тестовой выборки:", X_test.shape)
print("Размер y_train:", y_train.shape)
print("Размер y_test:", y_test.shape)

Размер обучающей выборки: (8000, 7)
Размер тестовой выборки: (2000, 7)
Размер y_train: (8000,)
Размер y_test: (2000,)


### 4. Обучите следующие ансамблевые модели:

#### Две модели группы бэггинга (бэггинг или случайный лес или сверхслучайные деревья);

In [16]:
from sklearn.ensemble import BaggingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

bagging = BaggingRegressor(
    n_estimators=100,
    random_state=42
)

bagging.fit(X_train, y_train)

y_pred = bagging.predict(X_test)

print("Bagging Regressor")
print("MAE =", mean_absolute_error(y_test, y_pred))
print("MSE =", mean_squared_error(y_test, y_pred))
print("R² =", r2_score(y_test, y_pred))

Bagging Regressor
MAE = 4.835109050000001
MSE = 42.52131849243501
R² = 0.8149147679336048


In [17]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

rf = RandomForestRegressor(
    n_estimators=100,
    random_state=42
)

rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)

print("Random Forest Regressor")
print("MAE =", mean_absolute_error(y_test, y_pred_rf))
print("MSE =", mean_squared_error(y_test, y_pred_rf))
print("R² =", r2_score(y_test, y_pred_rf))

Random Forest Regressor
MAE = 4.822650750000001
MSE = 42.53171516987501
R² = 0.8148695136582219


In [18]:
results = pd.DataFrame({
    'Model': ['Bagging Regressor', 'Random Forest Regressor'],
    'MAE': [
        mean_absolute_error(y_test, y_pred),
        mean_absolute_error(y_test, y_pred_rf)
    ],
    'MSE': [
        mean_squared_error(y_test, y_pred),
        mean_squared_error(y_test, y_pred_rf)
    ],
    'R2': [
        r2_score(y_test, y_pred),
        r2_score(y_test, y_pred_rf)
    ]
})

print(results)

                     Model       MAE        MSE        R2
0        Bagging Regressor  4.835109  42.521318  0.814915
1  Random Forest Regressor  4.822651  42.531715  0.814870


#### AdaBoost

In [19]:
from sklearn.ensemble import AdaBoostRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

ada = AdaBoostRegressor(
    n_estimators=100,
    random_state=42
)

ada.fit(X_train, y_train)

y_pred_ada = ada.predict(X_test)

print("AdaBoost Regressor")
print("MAE =", mean_absolute_error(y_test, y_pred_ada))
print("MSE =", mean_squared_error(y_test, y_pred_ada))
print("R² =", r2_score(y_test, y_pred_ada))

AdaBoost Regressor
MAE = 7.107608050704333
MSE = 64.80910032907602
R² = 0.7179013304454471


Посмотрим важность признаков

In [20]:
feature_importance = pd.DataFrame({
    'Feature': X.columns,
    'Importance': ada.feature_importances_
})

print(feature_importance.sort_values(
    by='Importance',
    ascending=False
))

                 Feature  Importance
6       placement_status    0.873112
0            study_hours    0.052957
4  assignments_completed    0.028669
5         previous_score    0.023035
1             attendance    0.011502
3         internet_usage    0.006860
2            sleep_hours    0.003866


#### Градиентный бустинг

In [22]:
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

gb = GradientBoostingRegressor(
    n_estimators=100,
    random_state=42
)

gb.fit(X_train, y_train)

y_pred_gb = gb.predict(X_test)

print("Gradient Boosting Regressor")
print("MAE =", mean_absolute_error(y_test, y_pred_gb))
print("MSE =", mean_squared_error(y_test, y_pred_gb))
print("R² =", r2_score(y_test, y_pred_gb))

Gradient Boosting Regressor
MAE = 4.962519406416998
MSE = 41.08537197412162
R² = 0.8211651031536722


In [23]:
feature_importance = pd.DataFrame({
    'Feature': X.columns,
    'Importance': gb.feature_importances_
})

print(feature_importance.sort_values(
    by='Importance',
    ascending=False
))

                 Feature  Importance
6       placement_status    0.743451
0            study_hours    0.109540
4  assignments_completed    0.062028
5         previous_score    0.043559
1             attendance    0.020899
3         internet_usage    0.011768
2            sleep_hours    0.008754


### 5. Оцените качество моделей с помощью одной из подходящих для задачи метрик. Сравните качество полученных моделей.

In [24]:
results = pd.DataFrame({
    'Model': [
        'Bagging Regressor',
        'Random Forest Regressor',
        'AdaBoost Regressor',
        'Gradient Boosting Regressor'
    ],
    'MAE': [
        mean_absolute_error(y_test, y_pred),
        mean_absolute_error(y_test, y_pred_rf),
        mean_absolute_error(y_test, y_pred_ada),
        mean_absolute_error(y_test, y_pred_gb)
    ],
    'MSE': [
        mean_squared_error(y_test, y_pred),
        mean_squared_error(y_test, y_pred_rf),
        mean_squared_error(y_test, y_pred_ada),
        mean_squared_error(y_test, y_pred_gb)
    ],
    'R2': [
        r2_score(y_test, y_pred),
        r2_score(y_test, y_pred_rf),
        r2_score(y_test, y_pred_ada),
        r2_score(y_test, y_pred_gb)
    ]
})

print(results)

                         Model       MAE        MSE        R2
0            Bagging Regressor  4.835109  42.521318  0.814915
1      Random Forest Regressor  4.822651  42.531715  0.814870
2           AdaBoost Regressor  7.107608  64.809100  0.717901
3  Gradient Boosting Regressor  4.962519  41.085372  0.821165


По результатам эксперимента наилучшее качество показала модель Gradient Boosting Regressor, которая получила наибольшее значение коэффициента детерминации (R² = 0.821) и наименьшее значение среднеквадратичной ошибки (MSE = 41.09). Это свидетельствует о наиболее точном прогнозировании экзаменационной оценки среди рассмотренных моделей.

Модели Bagging Regressor и Random Forest Regressor продемонстрировали практически одинаковые результаты (R² ≈ 0.815), немного уступая градиентному бустингу.

Наихудшие показатели показала модель AdaBoost Regressor (R² = 0.718, MAE = 7.11, MSE = 64.81), что говорит о меньшей эффективности данного алгоритма для выбранного набора данных.

Таким образом, для рассматриваемой задачи прогнозирования оценки экзамена наиболее эффективным оказался алгоритм градиентного бустинга, тогда как методы бэггинга показали хорошие, но несколько более слабые результаты, а AdaBoost значительно уступил остальным моделям по всем исследуемым метрикам.